# Демонстрационный запуск MOT-проекта

Короткий запуск baseline BoT-SORT и экспериментального RAFT-GMC на одной последовательности.

Полные результаты экспериментов: `results/experiments/`.


## 1. Параметры демонстрации


In [ ]:
REPO_URL = "https://github.com/AnixGG/MotFlow.git"
REPO_DIR = "/content/MOT"

DATA_GDRIVE_URL = "https://drive.google.com/file/d/1IG_6I8KSy3ZlQy0Hre_uL5xxIQge7Mv6/view?usp=sharing" 
DATA_ARCHIVE_NAME = "sportsmot_demo.zip"

MODEL_GDRIVE_URL = "https://drive.google.com/file/d/1Kjo-ei1KV7nVa_jmnVzB8YWVu9tp0BqW/view?usp=drive_link"

DEMO_SEQUENCE = ""
ENABLE_VISUALIZATION = True


## 2. Клонирование репозитория и установка зависимостей


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path(REPO_DIR)
if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "trackeval==1.3.0", "--no-deps"], check=True)

print("Репозиторий:", repo_dir)


## 3. Загрузка demo-данных и весов


In [ ]:
import shutil

work_dir = repo_dir / "demo_runtime"
work_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)

def download_gdrive(url, output_path):
    output_path = Path(output_path)
    subprocess.run(["gdown", "--fuzzy", url, "-O", str(output_path)], check=True)
    return output_path

data_archive = download_gdrive(DATA_GDRIVE_URL, work_dir / DATA_ARCHIVE_NAME)
model_path = download_gdrive(MODEL_GDRIVE_URL, work_dir / "yolo11_best.pt")

data_extract_dir = work_dir / "dataset"
if data_extract_dir.exists():
    shutil.rmtree(data_extract_dir)
shutil.unpack_archive(str(data_archive), str(data_extract_dir))

print("Данные распакованы:", data_extract_dir)
print("Веса модели:", model_path)


## 4. Проверка demo-split

Demo-архив содержит `val/<sequence>/img1` и `val/<sequence>/gt/gt.txt`.


In [ ]:
demo_root = data_extract_dir / "sportsmot_demo" / "dataset" / "val"

sequence_names = sorted(p.name for p in demo_root.iterdir() if p.is_dir())
DEMO_SEQUENCE = DEMO_SEQUENCE or sequence_names[0]

demo_seq = demo_root / DEMO_SEQUENCE
images = sorted((demo_seq / "img1").glob("*"))

print("Demo split:", demo_root)
print("Sequence:", DEMO_SEQUENCE)
print("Frames:", len(images))


## 5. Создание demo-конфигов

Baseline: `sparseOptFlow 4x`. Эксперимент: `RAFT-GMC small 128`.


In [ ]:
import yaml
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
config_dir = repo_dir / "configs" / "demo_runtime"
config_dir.mkdir(parents=True, exist_ok=True)

def make_config(output_path, outdir, gmc):
    cfg = {
        "data": str(demo_root),
        "sequences": [DEMO_SEQUENCE],
        "outdir": str(outdir),
        "device": device,
        "warmup_frames": 5,
        "model": str(model_path),
        "conf": 0.25,
        "iou": 0.7,
        "imgsz": 640,
        "classes": [0],
        "tracker_config": "configs/botsort_baseline.yaml",
        "visualization": {"enabled": bool(ENABLE_VISUALIZATION), "fps": 30},
    }
    if gmc == "sparse":
        cfg["gmc"] = "sparseOptFlow"
        cfg["gmc_downscale"] = 4
    elif gmc == "raft":
        cfg["gmc"] = "raft"
        cfg["raft_gmc"] = {
            "model": "small",
            "device": device,
            "mixed_precision": False,
            "scale_gmc": 1,
            "image_size": 128,
            "sample_step": 8,
            "ransac_reproj_threshold": 3.0,
        }
    with open(output_path, "w") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return output_path

baseline_config = make_config(config_dir / "demo_baseline.yaml", repo_dir / "experiments" / "demo" / "baseline_sparse4", "sparse")
raft_config = make_config(config_dir / "demo_raft.yaml", repo_dir / "experiments" / "demo" / "raft_small128", "raft")

print("Device:", device)
print("Baseline config:", baseline_config)
print("RAFT config:", raft_config)


## 6. Предзагрузка RAFT


In [ ]:
from torchvision.models.optical_flow import Raft_Small_Weights, raft_small

_ = raft_small(weights=Raft_Small_Weights.DEFAULT, progress=True).eval()
del _
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("RAFT-small weights cached")


## 7. Запуск baseline и RAFT-GMC


In [ ]:
%cd /content/MOT
!python3 -B scripts/run_baseline.py --config {baseline_config}


In [ ]:
%cd /content/MOT
!python3 -B scripts/run_experiment.py --config {raft_config}


## 8. Метрики и время


In [ ]:
import pandas as pd
from IPython.display import display

baseline_dir = repo_dir / "experiments" / "demo" / "baseline_sparse4"
raft_dir = repo_dir / "experiments" / "demo" / "raft_small128"

display(pd.concat([
    pd.read_csv(baseline_dir / "metrics.csv").assign(run="baseline"),
    pd.read_csv(raft_dir / "metrics.csv").assign(run="raft"),
]))

display(pd.concat([
    pd.read_csv(baseline_dir / "timing.csv").assign(run="baseline"),
    pd.read_csv(raft_dir / "timing.csv").assign(run="raft"),
]))


Визуализация <outdir>/vis/